# Extending PyTorch: custom layers, losses, and autograd Functions

_PyTorch (a complete course)_

**Write your own nn.Module layer with learnable nn.Parameters, a custom loss function, and — when you need a hand-written gradient — a torch.autograd.Function verified by gradcheck.**

PyTorch is extensible at three layers, and the trick is knowing how far down you must go.

       Most of the time you never touch the backward pass. If your idea is built from differentiable
       torch operations &mdash; matrix multiplies, additions, exponentials, comparisons that route gradients &mdash;
       then writing the forward is enough. Autograd recorded every op into a graph and will replay it
       in reverse to get exact gradients. A custom layer and a custom loss are exactly this: you supply the
       forward formula and learnable nn.Parameters, and the gradient comes for free.

---

This notebook is a practice scaffold for the **Extending PyTorch: custom layers, losses, and autograd Functions** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)                          # reproducible weights and data

# ============================================================
# 1. A CUSTOM LAYER: a from-scratch Linear.
#    Weights are nn.Parameter (so they TRAIN); a counter is a
#    register_buffer (non-learnable state that still saves/moves).
# ============================================================
class MyLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()                    # MUST be first
        # nn.Parameter -> registered + requires_grad=True -> the optimizer trains it.
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias   = nn.Parameter(torch.zeros(out_features))
        # A buffer: NON-learnable state. Moves with .to(device), saved in state_dict,
        # but never updated by the optimizer.
        self.register_buffer("call_count", torch.zeros(1))

    def forward(self, x):
        self.call_count += 1                  # buffer bookkeeping
        return x @ self.weight.t() + self.bias    # pure differentiable ops -> autograd backward for free

layer = MyLinear(4, 3)
print("params (trained):", [name for name, _ in layer.named_parameters()])
#   -> ['weight', 'bias']                     (call_count is NOT here)
print("buffers (not trained):", [name for name, _ in layer.named_buffers()])
#   -> ['call_count']
print("state_dict keys:", list(layer.state_dict().keys()))
#   -> ['weight', 'bias', 'call_count']       (buffer IS saved)

# ============================================================
# 2. A CUSTOM LOSS: just a function returning a scalar.
#    Built from differentiable torch ops -> autograd handles backward.
# ============================================================
def my_mse(pred, target):
    return ((pred - target) ** 2).mean()

pred   = torch.randn(8, 3)
target = torch.randn(8, 3)
print("my_mse vs nn.MSELoss:",
      float(my_mse(pred, target)),
      float(nn.MSELoss()(pred, target)))      # identical

# ============================================================
# 3. A CUSTOM AUTOGRAD FUNCTION: square, with a HAND-WRITTEN gradient.
#    Use this when you need to own the backward (non-standard /
#    more efficient gradient, or to wrap non-PyTorch code).
# ============================================================
class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)              # backward needs x for d/dx(x^2)=2x
        return x * x

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x            # chain rule: dL/dx = dL/dy * 2x

square = Square.apply                          # call via .apply, NOT Square.forward

# Verify the backward math. gradcheck needs float64 inputs + requires_grad.
gx = torch.randn(5, dtype=torch.float64, requires_grad=True)
ok = torch.autograd.gradcheck(square, (gx,), eps=1e-6, atol=1e-4)
print("gradcheck passed:", ok)                # -> True (our 2x backward is correct)

# ============================================================
# 4. TRAIN the custom layer with the custom loss for a few steps.
# ============================================================
torch.manual_seed(1)
X = torch.randn(64, 4)
true_W = torch.randn(3, 4); true_b = torch.randn(3)
Y = X @ true_W.t() + true_b                    # the target the layer should learn

model = MyLinear(4, 3)
opt = torch.optim.Adam(model.parameters(), lr=0.1)   # only weight & bias are passed
for step in range(200):
    opt.zero_grad()                            # clear old grads (they accumulate!)
    loss = my_mse(model(X), Y)                 # custom loss; autograd backprops it
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")
print("final loss:", round(my_mse(model(X), Y).item(), 5))   # near 0
print("forward calls counted by buffer:", int(model.call_count.item()))

## Visualize the data & results

_Is the hand-written backward of a custom autograd Function correct? For Square (f(x)=x^2), plot the analytic gradient our backward returns (2x) against a numerical finite-difference gradient across x — gradcheck's idea, by hand._

In [ ]:
import numpy as np

xs = np.linspace(-3, 3, 31)

# Analytic gradient our custom Square.backward returns:  d/dx (x^2) = 2x
analytic = 2 * xs

# Numerical gradient via central finite differences (what gradcheck does):
#   f'(x) ~= (f(x+h) - f(x-h)) / (2h)
h = 1e-4
f = lambda x: x ** 2
numerical = (f(xs + h) - f(xs - h)) / (2 * h)

print("max abs difference:", np.max(np.abs(analytic - numerical)))   # ~1e-11
for xv, a, n in zip(xs[::5], analytic[::5], numerical[::5]):
    print(f"x={xv:+.1f}  analytic={a:+6.3f}  numerical={n:+6.3f}")

import matplotlib.pyplot as plt
plt.plot(xs, analytic, color="#4ea1ff", label="analytic backward 2x", linewidth=3)
plt.plot(xs, numerical, color="#ff7b72", label="numerical (finite diff)", linestyle="--")
plt.xlabel("x"); plt.ylabel("df/dx")
plt.title("Custom Function backward (2x) vs numerical gradient of f(x) = x^2")
plt.legend(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You write a custom layer with self.weight = torch.randn(8, 4) and self.bias = torch.zeros(8). Training runs without error but the layer's weights never change and the loss is stuck. What is the bug, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the weights are plain tensors, not nn.Parameter. — _Only nn.Parameter (or submodules) assigned to self get registered; a bare tensor is invisible to the module._
- Check whether they appear in model.parameters(). — _If they are missing from parameters(), the optimizer was never handed them, so it cannot update them._
- Wrap them: self.weight = nn.Parameter(torch.randn(8, 4)) and self.bias = nn.Parameter(torch.zeros(8)). — _nn.Parameter registers the tensor and sets requires_grad=True, so it lands in parameters() and trains._

**Answer:** The weights are plain tensors, so they are never registered &mdash; they are absent from model.parameters(), the optimizer never receives them, and they never update (a bare tensor also has requires_grad=False by default). Wrap them in nn.Parameter: self.weight = nn.Parameter(torch.randn(8, 4)). Now they register, require grad, and train. Use register_buffer only for state you do not want trained.

</details>

**Problem 2.** You wrote a torch.autograd.Function for a custom op and want to be sure the backward is correct before training a real model on it. How do you check it, and what setup does the check require?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use torch.autograd.gradcheck(Fn.apply, inputs). — _gradcheck compares your analytic backward against a numerical finite-difference gradient, which catches a wrong derivative formula._
- Make the input tensors float64 (.double()) with requires_grad=True. — _Finite differences subtract nearly equal numbers; 64-bit precision is needed or the check reports false mismatches._
- Call the op through .apply, never Fn.forward directly. — _.apply records the op on the autograd graph so backward actually runs; calling forward bypasses autograd._

**Answer:** Run torch.autograd.gradcheck(MyFn.apply, (x,)) where x is a float64 tensor with requires_grad=True. gradcheck perturbs each input by a tiny step, measures the numerical slope, and compares it to what your backward returns; it returns True (or raises) so you learn about a bad gradient before it quietly ruins training. The double precision is essential &mdash; finite differences lose too many digits in float32.

</details>

**Problem 3.** Your model needs a fixed lookup table (a positional-encoding matrix) that should travel to the GPU with the model and be saved in checkpoints, but must not be updated by the optimizer. Where do you put it, and why not just make it an attribute or a parameter?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reject a plain attribute (self.table = tensor). — _A plain tensor does not move with model.to(device) and is not in state_dict(), so it stays on the CPU and is lost on save/load._
- Reject nn.Parameter. — _A Parameter would be trained by the optimizer, which you explicitly do not want for a fixed table._
- Use self.register_buffer("table", tensor). — _A buffer is registered non-learnable state: it moves with the module and saves in the checkpoint, but the optimizer ignores it._

**Answer:** Register it as a buffer: self.register_buffer("table", tensor). A buffer is the right home for non-learnable state &mdash; it walks the module tree with model.to(device) and appears in state_dict() (so it is checkpointed), yet it is not returned by model.parameters(), so the optimizer never touches it. A plain attribute would be left behind on device moves and saves; an nn.Parameter would wrongly be trained.

</details>